In [43]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

def show_df(df):
    print(df.shape)
    display(pd.concat([df.head(2), df.tail(1)]))

# Data Cleaning

Données sources : [Catalogue of Meteorites (MetCat) - Natural History Museum](https://data.nhm.ac.uk/dataset/metcat/resource/96dc3c09-49fd-4af6-b2fb-5a48a76d09ee?view_id=7a7b270c-23fc-4237-8501-451b1bc5c6f8)

In [31]:
data_path = "data/resource.csv"

df = pd.read_csv(data_path)
show_df(df)

(26474, 27)


,(A)chondrite,_id,Bandwidth (mm),Class,Country,Day,Decimal latitude,Decimal longitude,Find or fall,Group,Hour,Locality,MetCat key,Month,Name,NHM specimen data,Petrologic type,Present in NHM collection,Recovered weight,Shock stage,Synonyms,Type,Validity,Verbatim latitude,Verbatim longitude,Weathering grade,Year
0,Chondrite,26478,NaN,Ordinary,England,4,51.7833,-1.7833,Fall,LL,1630,Gloucestershire,A1030,August,Aldsworth,https://data.nhm.ac.uk/dataset/collection-spec...,5,Y,0.7 kg,NaN,Cirencester,Stone,Valid,51° 47' N,1° 47' W,NaN,1835
1,NaN,26481,2,Coarse octahedrite,USA,NaN,35.7500,-81.2500,Find,IAB,NaN,North Carolina,A1090,NaN,Alexander County,NaN,Og,NaN,NaN,NaN,Cedar Creek,Iron,Valid,35° 45' N,81° 15' W,NaN,1875
26473,Chondrite,52942,NaN,Ordinary,Malawi,25,-15.1833,35.2833,Fall,L,745,Southern province,Z510,January,Zomba,https://data.nhm.ac.uk/dataset/collection-spec...,6,Y,7.5 kg,NaN,Mount Zomba,Stone,Valid,15° 11' S,35° 17' E,NaN,1899


In [116]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26474 entries, 0 to 26473
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   (A)chondrite               19469 non-null  object 
 1   _id                        26474 non-null  int64  
 2   Bandwidth (mm)             537 non-null    object 
 3   Class                      20228 non-null  object 
 4   Country                    26200 non-null  object 
 5   Day                        2454 non-null   object 
 6   Decimal latitude           26050 non-null  float64
 7   Decimal longitude          26050 non-null  float64
 8   Find or fall               26414 non-null  object 
 9   Group                      20224 non-null  object 
 10  Hour                       829 non-null    object 
 11  Locality                   23183 non-null  object 
 12  MetCat key                 26474 non-null  object 
 13  Month                      3546 non-null   obj

In [ ]:
def process_range(val):
    if "-" in val:
        res = val.split("-")
        res = (float(res[0]) + float(res[1])) / 2
        return res
    return float(val)

def clean_weight(val):
    try:
        val_split = val
        for preffix in ["> ", ">", "~ ", "~"]:
            val_split = val_split.strip(preffix)
        val_split = val_split.split(" ")
        unit = val_split[-1]
        if unit in ["g", "G"]:
            return process_range(val_split[0]) / 1000
        if unit == "kg":
            return process_range(val_split[0])
        if unit in ["0", "missing"]:
            return np.nan
        if unit == "t":
            return process_range(val_split[0]) * 1000
        if unit == "mg":
            return process_range(val_split[0]) / 1000000
        if ".kg" in unit:
            return float(unit.split(".kg")[0])
        if "kg" in unit:
            return float(unit.split("kg")[0])
        return np.nan
    except:
        print(val, val_split)

def clean_year(yr):
    try:
        return int(yr.strip("s").strip("?"))
    except:
        try:
            yr_split = yr.split("-")
            if len(yr_split[-1]) == 4:
                yr_l = int(yr_split[0])
                yr_r = int(yr_split[1])
            else:
                yr_l = int(yr_split[0])
                yr_r = int(yr_split[1][:2] + yr_split[1])
            return (int(yr_l) + int(yr_r)) // 2
        except:
            return np.nan

def clean_type(val):
    if val == "Stony-iron":
        return "Stony-Iron"
    return val

In [113]:
df_clean = df.copy()
df_clean["Recovered weight"] = df_clean["Recovered weight"].fillna("0").apply(clean_weight)
df_clean["Year"] = df_clean["Year"].apply(clean_year)
df_clean["Type"] = df_clean["Type"].apply(clean_type)
show_df(df_clean)

(26474, 27)


,(A)chondrite,_id,Bandwidth (mm),Class,Country,Day,Decimal latitude,Decimal longitude,Find or fall,Group,Hour,Locality,MetCat key,Month,Name,NHM specimen data,Petrologic type,Present in NHM collection,Recovered weight,Shock stage,Synonyms,Type,Validity,Verbatim latitude,Verbatim longitude,Weathering grade,Year
0,Chondrite,26478,NaN,Ordinary,England,4,51.7833,-1.7833,Fall,LL,1630,Gloucestershire,A1030,August,Aldsworth,https://data.nhm.ac.uk/dataset/collection-spec...,5,Y,0.7,NaN,Cirencester,Stone,Valid,51° 47' N,1° 47' W,NaN,1835.0
1,NaN,26481,2,Coarse octahedrite,USA,NaN,35.7500,-81.2500,Find,IAB,NaN,North Carolina,A1090,NaN,Alexander County,NaN,Og,NaN,NaN,NaN,Cedar Creek,Iron,Valid,35° 45' N,81° 15' W,NaN,1875.0
26473,Chondrite,52942,NaN,Ordinary,Malawi,25,-15.1833,35.2833,Fall,L,745,Southern province,Z510,January,Zomba,https://data.nhm.ac.uk/dataset/collection-spec...,6,Y,7.5,NaN,Mount Zomba,Stone,Valid,15° 11' S,35° 17' E,NaN,1899.0


In [ ]:
save_path = "data/resource_clean.csv"

df_clean.to_csv(save_path)